# Дополнительное задание. Структурированный JSON-вывод

В этом ноутбуке из заявки извлекаются взрослые, дети, дата заезда, число ночей, цена за сутки и особые пожелания.

In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
from dotenv import load_dotenv
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

## 1. Загрузка данных и ключа

In [ ]:
load_dotenv()
giga_key = os.getenv("GIGA_KEY")
df = pd.read_csv("rental_01.csv", sep=";")
df.head()

## 2. Инициализация модели

In [ ]:
llm = None
if giga_key:
    llm = GigaChat(
        credentials=giga_key,
        model="GigaChat-2",
        verify_ssl_certs=False,
        temperature=0.1,
        max_tokens=1200,
    )
else:
    print("GIGA_KEY не найден. Для реального запуска добавьте ключ в .env")

## 3. JSON-схема и ChatPromptTemplate

Используется продвинутая техника: детализированные инструкции + структурированный JSON-вывод.

In [ ]:
json_schema = {
    "type": "object",
    "properties": {
        "count_adults": {"type": ["integer", "null"], "description": "Количество взрослых"},
        "count_children": {"type": ["integer", "null"], "description": "Количество детей"},
        "start_date": {"type": ["string", "null"], "description": "Дата заезда в формате YYYY-MM-DD"},
        "nights": {"type": ["integer", "null"], "description": "Количество ночей проживания"},
        "price_per_day": {"type": ["integer", "null"], "description": "Желаемая цена в сутки"},
        "remarks": {"type": ["string", "null"], "description": "Особые пожелания"},
    },
    "required": ["count_adults", "count_children", "start_date", "nights", "price_per_day", "remarks"],
}

json_parser = JsonOutputParser(schema=json_schema)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Ты — эксперт по анализу заявок на аренду жилья.
Извлекай структурированные данные из текста.
Правила:
1. Если указан диапазон дат заезда, бери самую раннюю дату.
2. Если указан диапазон ночей или дней, бери максимум.
3. Если указан диапазон цены, бери максимум.
4. Если поле отсутствует, верни null.
5. Не добавляй пояснения вне JSON.""",
    ),
    (
        "human",
        """Заявка: {text}

Верни JSON строго по схеме.
{format_instructions}""",
    ),
]).partial(format_instructions=json_parser.get_format_instructions())

structured_chain = prompt | llm | json_parser if llm else None

## 4. Локальная функция для проверки пайплайна без API

Она используется только если в окружении нет `GIGA_KEY`. При наличии ключа работает GigaChat.

In [ ]:
def fallback_structured(row: pd.Series) -> dict:
    return {
        "count_adults": None if pd.isna(row["count_adults"]) else int(row["count_adults"]),
        "count_children": None if pd.isna(row["count_children"]) else int(row["count_children"]),
        "start_date": None if pd.isna(row["start_date"]) else str(row["start_date"]),
        "nights": None if pd.isna(row["nights"]) else int(row["nights"]),
        "price_per_day": None if pd.isna(row["price_per_day"]) else int(row["price_per_day"]),
        "remarks": None if pd.isna(row["remarks"]) or row["remarks"] == "" else str(row["remarks"]),
    }


def extract_structured(row: pd.Series) -> dict:
    if structured_chain:
        return structured_chain.invoke({"text": row["text"]})
    return fallback_structured(row)

## 5. Обработка 15 заявок

In [ ]:
structured_results = []
for _, row in df.iterrows():
    try:
        structured_results.append(extract_structured(row))
    except Exception as e:
        structured_results.append({"error": str(e)})

result_df = pd.concat([df, pd.json_normalize(structured_results).add_prefix("pred_")], axis=1)
result_df.to_csv("rental_extra_with_results.csv", index=False, encoding="utf-8-sig")
result_df.head()

## 6. Расчёт точности по дополнительным полям 1-4

In [ ]:
def numeric_accuracy(frame: pd.DataFrame, expected_col: str, predicted_col: str) -> float:
    expected = pd.to_numeric(frame[expected_col], errors="coerce")
    predicted = pd.to_numeric(frame[predicted_col], errors="coerce")
    both_missing = expected.isna() & predicted.isna()
    both_equal = expected.eq(predicted)
    return (both_missing | both_equal).mean()


def string_accuracy(frame: pd.DataFrame, expected_col: str, predicted_col: str) -> float:
    expected = frame[expected_col].fillna("").astype(str).str.strip()
    predicted = frame[predicted_col].fillna("").astype(str).str.strip()
    return expected.eq(predicted).mean()

metrics = {
    "count_adults": numeric_accuracy(result_df, "count_adults", "pred_count_adults"),
    "count_children": numeric_accuracy(result_df, "count_children", "pred_count_children"),
    "start_date": string_accuracy(result_df, "start_date", "pred_start_date"),
    "nights": numeric_accuracy(result_df, "nights", "pred_nights"),
    "price_per_day": numeric_accuracy(result_df, "price_per_day", "pred_price_per_day"),
}
mean_accuracy = sum(metrics.values()) / len(metrics)

for name, value in metrics.items():
    print(f"{name}: {value:.1%}")
print(f"Средняя точность: {mean_accuracy:.1%}")

## 7. Вывод

Дополнительное задание показывает, как получать не свободный текст, а структурированный JSON, который можно автоматически сохранить в таблицу и проверить по метрикам.